In [ ]:
import argparse
import torch
import numpy as np
import random
from peft import (
    LoraConfig,
    get_peft_model
)
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
import os
import sys
import json
import transformers
import warnings
from datasets import load_dataset
from predict_module import sft_dataloader

from utils.prompts import PREDICT_INSTRUCTION
from utils.fewshots import PREDICT_EXAMPLES

# Thiết lập seed
fix_seed = 100
random.seed(fix_seed)
torch.manual_seed(fix_seed)
np.random.seed(fix_seed)

# Cấu hình tham số cho huấn luyện
args = argparse.Namespace(
    
    wandb=False,  # Tắt logging với Weights & Biases
    data_path="./data/DeepSeekLLM_top1_stock_merge_sample.json",  # Đường dẫn file dữ liệu
    output_path="./saved_models/lora-DeepSeek-R1-Distill-Qwen",  # Thư mục lưu mô hình LoRA
    model_path="deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B",  # Mô hình DeepSeek
    eval_steps=200,  # Số bước đánh giá
    save_steps=200,  # Số bước lưu checkpoint
    resume_from_supervised_checkpoint=None,  # Không resume từ checkpoint
    ignore_data_skip="False",  # Không bỏ qua dữ liệu khi resume
    num_reflect_trials=2,  # Số lần thử phản ánh
    datasets_dir="./datasets/",  # Thư mục datasets
    local_rank=0,  # Rank cục bộ cho DDP
    resume_from_reward_checkpoint=False,  # Không resume từ reward checkpoint
    deepspeed=None,  # Không dùng DeepSpeed
    per_device_train_batch_size=4,  # Batch size huấn luyện trên mỗi GPU
    per_device_eval_batch_size=4,  # Batch size đánh giá trên mỗi GPU
    reward_gradient_accumulation_steps=8,  # Số bước tích lũy gradient cho reward
    reward_learning_rate=3e-5,  # Learning rate cho reward
    weight_decay=0.001,  # Trọng số giảm dần
    reward_base_model="deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B",  # Mô hình reward
    bf16=False,  # Sử dụng fp16 thay vì bf16
    num_train_epochs=2,  # Số epoch huấn luyện
    train_subset=100000,  # Số mẫu huấn luyện
    eval_subset=50000,  # Số mẫu đánh giá
    gradient_checkpointing=True,  # Bật gradient checkpointing để tiết kiệm VRAM
    optim="adamw_torch",  # Optimizer AdamW từ PyTorch
    lr_scheduler_type="cosine",  # Lịch trình learning rate kiểu cosine
    reward_adapter="./saved_models/reward_model_deepseek-r1-distill-qwen",  # Adapter reward
    rl_base_model="./saved_models/lora-DeepSeek-R1-Distill-Qwen-adapter-merged",  # Mô hình RL
    tokenizer_name="deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B",  # Tokenizer
    reward_model_name="./saved_models/reward_model_deepseek-r1-distill-qwen-adapter-merged",  # Mô hình reward merged
    log_with=None,  # Không dùng logging cụ thể
    rl_learning_rate=2e-5,  # Learning rate cho RL
    output_max_length=256,  # Độ dài đầu ra tối đa
    mini_batch_size=4,  # Kích thước mini-batch
    batch_size=128,  # Kích thước batch tổng
    ppo_epochs=4,  # Số epoch cho PPO
    rl_gradient_accumulation_steps=32,  # Số bước tích lũy gradient cho RL
    adafactor=False,  # Không dùng Adafactor
    early_stopping=True,  # Bật early stopping
    target_kl=0.1,  # KL target cho RL
    reward_baseline=0,  # Baseline cho reward
    batched_gen=True,  # Tạo batch
    save_freq=100,  # Tần suất lưu
    output_dir="./saved_models/tuning_deepseek_r1_distill_qwen_checkpoints/",  # Thư mục lưu checkpoint
    seed=0,  # Seed cho RL
    num_shots=4,  # Số shots cho few-shot
    save_dir="results/"  # Thư mục lưu kết quả
)


print("Args in experiment:")
print(args)

args.data_path


## PREDICT
- SFT
- train reward model
- PPO (tuning with rl)

In [ ]:
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    TrainingArguments,
    Trainer,
)
# from trl import SFTTrainer
import torch
from peft import LoraConfig, get_peft_model, set_peft_model_state_dict


def supervised_finetune(args):
    # --- Các hằng số huấn luyện ---
    MICRO_BATCH_SIZE = args.per_device_train_batch_size  # Batch size mỗi GPU
    BATCH_SIZE = args.batch_size  # Batch size tổng
    MAX_STEPS = None  # Số bước tối đa, tính động
    GRADIENT_ACCUMULATION_STEPS = BATCH_SIZE // MICRO_BATCH_SIZE  # Bước tích lũy gradient
    EPOCHS = args.num_train_epochs  # Số epoch
    LEARNING_RATE = 3e-4  # Tốc độ học
    CUTOFF_LEN = 256  # Độ dài chuỗi tối đa
    LORA_R = 16  # Rank LoRA
    LORA_ALPHA = 32  # Hệ số scale LoRA
    LORA_DROPOUT = 0.05  # Dropout LoRA
    VAL_PCT = 0.1  # Tỷ lệ validation
    TARGET_MODULES = ["q_proj", "k_proj", "v_proj", "o_proj"]  # Layer áp dụng LoRA
    DATA_PATH = args.data_path  # Đường dẫn dữ liệu
    OUTPUT_DIR = args.output_path  # Thư mục lưu mô hình
    world_size = int(os.environ.get("WORLD_SIZE", 1))  # Số GPU (DDP)


    # --- Xử lý DDP ---
    ddp = world_size != 1  # Kiểm tra đa GPU
    if ddp:
        torch.cuda.set_device(int(os.environ.get("LOCAL_RANK", 0)))  # Gán GPU
        GRADIENT_ACCUMULATION_STEPS = GRADIENT_ACCUMULATION_STEPS // world_size  # Chia tích lũy gradient

    #==============================================================================================================================
    print(f"Đang tải mô hình từ: {args.model_path}")  # In đường dẫn mô hình

    # Step 3: Load model and tokenizer
    tokenizer = AutoTokenizer.from_pretrained(
        args.model_path,
        add_eos_token=True,  # Thêm token kết thúc
        local_files_only=args.offline if hasattr(args, 'offline') else False  # Chế độ offline
        )

    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token  # Gán pad_token

    model = AutoModelForCausalLM.from_pretrained(
        args.model_path, 
        torch_dtype=torch.float16, 
        device_map="auto")

    # --- Tải dữ liệu ---
    dataset = load_dataset("json", data_files=DATA_PATH)  # Tải JSON dataset


    val_set_size = int(VAL_PCT * len(dataset["train"]))  # Tính validation size
    print(dataset)  # In thông tin dataset


    # --- Tải dữ liệu huấn luyện và đánh giá ---
    dataloader = sft_dataloader.SFTDataLoader(dataset, CUTOFF_LEN, val_set_size, tokenizer)  # Định dạng và tokenize
    train_data, val_data = dataloader.load_data()  # Chia train/validation
    train_data, val_data

    # Step 4: Configure LoRA
    peft_config = LoraConfig(
        r=LORA_R,  # Rank LoRA
        lora_alpha=LORA_ALPHA,  # Scale LoRA
        target_modules=TARGET_MODULES,  # Layer LoRA
        lora_dropout=LORA_DROPOUT,  # Dropout
        bias="none",  # Không bias
        task_type="CAUSAL_LM"  # Tác vụ ngôn ngữ
    )


    model = get_peft_model(model, peft_config)


    # --- Tính max_steps ---
    now_max_steps = max((len(dataset["train"]) - val_set_size) // BATCH_SIZE * EPOCHS, EPOCHS)  # Số bước tối đa


    # --- Xử lý checkpoint ---
    if args.resume_from_supervised_checkpoint:  # Nếu có checkpoint
        checkpoint_name = os.path.join(args.resume_from_supervised_checkpoint, "pytorch_model.bin")  # Đường dẫn checkpoint
        if not os.path.exists(checkpoint_name):
            pytorch_bin_path = checkpoint_name
            checkpoint_name = os.path.join(args.resume_from_supervised_checkpoint, "adapter_model.bin")  # Kiểm tra file khác
            if os.path.exists(checkpoint_name):
                os.rename(checkpoint_name, pytorch_bin_path)  # Đổi tên
                warnings.warn("Đã đổi tên 'adapter_model.bin' thành 'pytorch_model.bin'")
            else:
                args.resume_from_supervised_checkpoint = None  # Bỏ resume
        if os.path.exists(checkpoint_name):
            print(f"Tiếp tục từ: {checkpoint_name}")
            adapters_weights = torch.load(checkpoint_name)  # Tải LoRA
            model = set_peft_model_state_dict(model, adapters_weights)  # Áp dụng
        else:
            print(f"Không tìm thấy: {checkpoint_name}")
        train_args_path = os.path.join(args.resume_from_supervised_checkpoint, "trainer_state.json")  # File trạng thái
        if os.path.exists(train_args_path):
            base_train_args = json.load(open(train_args_path, 'r'))
            base_max_steps = base_train_args["max_steps"]  # Số bước cũ
            resume_scale = base_max_steps / now_max_steps
            if base_max_steps > now_max_steps:
                warnings.warn(f"Thay epoch {EPOCHS} bằng {base_max_steps}")
                EPOCHS = None
                MAX_STEPS = base_max_steps
            else:
                MAX_STEPS = now_max_steps
    else:
        MAX_STEPS = now_max_steps


    # Step 5: Define training arguments
    training_args = TrainingArguments(
        output_dir=OUTPUT_DIR,  # Directory to save results
        per_device_train_batch_size=MICRO_BATCH_SIZE,  # Batch size GPU
        gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,  # Tích lũy gradient
        warmup_steps=100,  # Bước khởi động
        num_train_epochs=EPOCHS if EPOCHS else 1,  # Epoch
        max_steps=MAX_STEPS,  # Số bước tối đa
        learning_rate=LEARNING_RATE,  # Tốc độ học
        bf16=args.bf16,  # BF16
        fp16=not args.bf16,  # FP16
        logging_steps=20,  # Log mỗi 20 bước
        eval_strategy="steps" if val_set_size > 0 else "no",  # Đánh giá
        save_strategy="steps",  # Lưu checkpoint
        eval_steps=args.eval_steps if val_set_size > 0 else None,  # Bước đánh giá
        save_steps=args.save_steps,  # Bước lưu
        save_total_limit=30,  # Số checkpoint tối đa
        load_best_model_at_end=True if val_set_size > 0 else False,  # Tải mô hình tốt
        ddp_find_unused_parameters=False if ddp else None,  # Tối ưu DDP
        report_to="wandb" if args.wandb else [],  # Báo cáo WandB
        optim=args.optim,  # Bộ tối ưu
        lr_scheduler_type=args.lr_scheduler_type,  # Scheduler
        remove_unused_columns=True,  # Xóa cột thừa
        max_grad_norm=1.0,  # Giới hạn gradient
    )


    # Step 6: Initialize the trainer
    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_data, 
        eval_dataset=val_data,  # Small evaluation set
        data_collator = transformers.DataCollatorForLanguageModeling(tokenizer, mlm=False),
    )

    # Step 7: Train the model
    trainer.train(resume_from_checkpoint=args.resume_from_supervised_checkpoint)
    
    model.save_pretrained(OUTPUT_DIR) 

# --- Chạy huấn luyện ---

supervised_finetune(args)

from predict_module.merge_peft_adapter import merge_peft_adapter

# --- Gộp adapter LoRA ---
merge_peft_adapter(model_name=args.output_path, output_name=args.rl_base_model)


# Giải phóng bộ nhớ trên GPU
import torch
import gc
torch.cuda.ipc_collect()
torch.cuda.empty_cache()
import gc
gc.collect()


In [ ]:
import torch
import torch.nn as nn
from transformers import (
    AutoConfig,
    AutoModelForSequenceClassification,
    AutoTokenizer,
    HfArgumentParser,
    PreTrainedTokenizerBase,
    Trainer,
    TrainingArguments,
)
from peft import LoraConfig, TaskType, get_peft_model
import evaluate
import numpy as np
from dataclasses import dataclass
from typing import Any, Dict, List, Optional, Union
from transformers.utils import PaddingStrategy

from predict_module import rm_dataloader

def train_reward_model(args):
    script_args = args
    dataset_name = script_args.datasets_dir
    print("dataset_name:", dataset_name)
    
    output_name = script_args.reward_adapter
    
    training_args = TrainingArguments(
        output_dir=output_name,
        learning_rate=script_args.reward_learning_rate,
        per_device_train_batch_size=1,  # Giảm batch size
        per_device_eval_batch_size=1,
        num_train_epochs=script_args.num_train_epochs,
        weight_decay=script_args.weight_decay,
        eval_strategy="steps",
        eval_steps=200,
        save_strategy="steps",
        save_steps=200,
        save_total_limit=2,
        gradient_accumulation_steps=8,  # Tăng gradient accumulation
        gradient_checkpointing=True,  # Tắt gradient checkpointing
        deepspeed=None,  # Tắt DeepSpeed để kiểm tra
        remove_unused_columns=False,
        label_names=[],
        logging_strategy="steps",
        logging_steps=10,
        optim=script_args.optim,
        lr_scheduler_type=script_args.lr_scheduler_type,
        report_to="none",
        no_cuda=False,  # Đảm bảo dùng GPU
        bf16=True,
    )
    
    # Load tokenizer and model
    tokenizer = AutoTokenizer.from_pretrained(script_args.reward_base_model, trust_remote_code=True)
    model = AutoModelForSequenceClassification.from_pretrained(
        script_args.reward_base_model,
        num_labels=1,
        torch_dtype=torch.bfloat16,  # Thử bfloat16
        trust_remote_code=True,
    )
    
    # Handle pad token
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
        model.config.pad_token_id = tokenizer.eos_token_id
    else:
        model.config.pad_token_id = tokenizer.pad_token_id
    
    # Set device
    # device_map = "cuda:0" if torch.cuda.is_available() else "cpu"
    # print("device_map:", device_map)
    # model = model.to(device_map)  # Chuyển thủ công
    
    # LoRA config
    peft_config = LoraConfig(
        task_type=TaskType.SEQ_CLS,
        inference_mode=False,
        r=2,  # Giảm r
        lora_alpha=4,
        lora_dropout=0.05,
        bias="none",
    )
    model = get_peft_model(model, peft_config)
    model.print_trainable_parameters()
    
    num_proc = 1
    reward_dataloder = rm_dataloader.RewardDataLoader(dataset_name, script_args.train_subset, script_args.eval_subset, num_proc, tokenizer)
    train_dataset, eval_dataset = reward_dataloder.load_data()
    
    @dataclass
    class RewardDataCollatorWithPadding:
        tokenizer: PreTrainedTokenizerBase
        padding: Union[bool, str, PaddingStrategy] = True
        max_length: Optional[int] = None
        pad_to_multiple_of: Optional[int] = None
        return_tensors: str = "pt"

        def __call__(self, features: List[Dict[str, Any]]) -> Dict[str, Any]:
            features_j = []
            features_k = []
            for feature in features:
                features_j.append(
                    {
                        "input_ids": feature["input_ids_j"],
                        "attention_mask": feature["attention_mask_j"],
                    }
                )
                features_k.append(
                    {
                        "input_ids": feature["input_ids_k"],
                        "attention_mask": feature["attention_mask_k"],
                    }
                )
            batch_j = self.tokenizer.pad(
                features_j,
                padding=self.padding,
                max_length=self.max_length,
                pad_to_multiple_of=self.pad_to_multiple_of,
                return_tensors=self.return_tensors,
            )
            batch_k = self.tokenizer.pad(
                features_k,
                padding=self.padding,
                max_length=self.max_length,
                pad_to_multiple_of=self.pad_to_multiple_of,
                return_tensors=self.return_tensors,
            )
            batch = {
                "input_ids_j": batch_j["input_ids"],
                "attention_mask_j": batch_j["attention_mask"].to(dtype=torch.bfloat16),
                "input_ids_k": batch_k["input_ids"],
                "attention_mask_k": batch_k["attention_mask"].to(dtype=torch.bfloat16),
                "return_loss": True,
            }
            return batch
    
    accuracy = evaluate.load("accuracy")
    
    def compute_metrics(eval_pred):
        predictions, _ = eval_pred
        predictions = np.argmax(predictions, axis=0)
        labels = np.zeros(predictions.shape)
        return accuracy.compute(predictions=predictions, references=labels)
    
    class RewardTrainer(Trainer):
        def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
            rewards_j = model(
                input_ids=inputs["input_ids_j"], attention_mask=inputs["attention_mask_j"])[0]
            rewards_k = model(
                input_ids=inputs["input_ids_k"], attention_mask=inputs["attention_mask_k"])[0]
            loss = -nn.functional.logsigmoid(rewards_j - rewards_k).mean()
            if return_outputs:
                return loss, {"rewards_j": rewards_j, "rewards_k": rewards_k}
            return loss
    
    trainer = RewardTrainer(
        model=model,
        args=training_args,
        train_dataset=train_dataset,
        eval_dataset=eval_dataset,
        compute_metrics=compute_metrics,
        data_collator=RewardDataCollatorWithPadding(
            tokenizer=tokenizer, max_length=256, pad_to_multiple_of=8),
    )
    
    model.config.use_cache = True
    trainer.train(script_args.resume_from_reward_checkpoint)
    
    print("Saving last checkpoint of the model")
    model.save_pretrained(output_name)

train_reward_model(args)

from predict_module.merge_peft_adapter import merge_peft_adapter
merge_peft_adapter(model_name=args.reward_adapter, output_name=args.reward_model_name)


# Giải phóng bộ nhớ trên GPU
import torch
import gc
torch.cuda.ipc_collect()
torch.cuda.empty_cache()
import gc
gc.collect()

In [ ]:
from dataclasses import dataclass, field
from typing import Optional
import torch
from accelerate import Accelerator
from datasets import load_dataset, concatenate_datasets
from peft import LoraConfig
from tqdm import tqdm
from transformers import Adafactor, AutoTokenizer, AutoModelForSequenceClassification, AutoConfig, AutoModelForCausalLM, DataCollatorWithPadding
from transformers import GenerationConfig, pipeline
from trl import AutoModelForCausalLMWithValueHead, PPOConfig, PPOTrainer
from trl.core import LengthSampler
from trl import create_reference_model
import os
import gc

tqdm.pandas()

def tuning_lm_with_rl(args):
    # Khởi tạo script_args từ args
    script_args = args
    reward_model_name = script_args.reward_model_name
    print("reward_model_name:", reward_model_name)

    # Đường dẫn dataset
    dataset_name = script_args.datasets_dir
    print("dataset_name:", dataset_name)

    # Cấu hình PPO
    config = PPOConfig(
        learning_rate=script_args.rl_learning_rate,
        # batch_size=script_args.batch_size,
        batch_size=4,
        # mini_batch_size=script_args.mini_batch_size,
        mini_batch_size=2,
        # gradient_accumulation_steps=script_args.rl_gradient_accumulation_steps,
        gradient_accumulation_steps=1,
        ppo_epochs=script_args.ppo_epochs,  
        seed=script_args.seed,
    )

    # Tên mô hình gốc
    model_name = script_args.rl_base_model
    print("model_name:", model_name)

    # Tải dataset
    train_dataset = load_dataset(dataset_name, split="train")
    print("train_dataset size:", len(train_dataset))

    # Cấu hình tham số cho sentiment pipeline
    sent_kwargs = {
        "return_all_scores": True,
        "function_to_apply": "none",
        "batch_size": 1,
        "truncation": True
    }

    # Tải tokenizer
    tokenizer = AutoTokenizer.from_pretrained(script_args.tokenizer_name, trust_remote_code=True)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    print("Tokenizer loaded:", tokenizer.__class__.__name__)

    def build_dataset(tokenizer, dataset_name, input_min_text_length=2, input_max_text_length=8):
        """
        Tạo dataset cho huấn luyện PPO.
        
        Args:
            tokenizer: Tokenizer để mã hóa văn bản.
            dataset_name: Tên hoặc đường dẫn dataset.
            input_min_text_length: Độ dài tối thiểu của câu hỏi.
            input_max_text_length: Độ dài tối đa của câu hỏi.
        
        Returns:
            Dataset đã được xử lý với các cột query và input_ids.
        """
        ds = load_dataset(dataset_name, split="train")
        
        original_columns = ds.column_names

        def preprocess_function(examples):
            new_examples = {
                "query": [],
                "input_ids": [],
            }
            # Giả định dataset có cột 'user_input' hoặc 'question'
            input_key = "user_input" if "user_input" in examples else "question"
            for question in examples[input_key]:
                query = "Question: " + question + "\n\nAnswer: "
                tokenized_question = tokenizer(
                    query,
                    truncation=True,
                    max_length=256,
                    return_tensors="pt"
                )
                new_examples["query"].append(query)
                new_examples["input_ids"].append(tokenized_question["input_ids"].squeeze(0))
            return new_examples

        ds = ds.map(
            preprocess_function,
            batched=True,
            num_proc=1,
            remove_columns=original_columns,
        )
        ds.set_format(type="torch")
        return ds

    # Tạo dataset
    dataset = build_dataset(tokenizer, dataset_name=dataset_name)
    print("Dataset created with", len(dataset), "samples")

    def collator(data):
        """
        Collator để xử lý batch dữ liệu.
        """
        return dict((key, [d[key] for d in data]) for key in data[0])
    

    # Đặt seed để đảm bảo tính tái lập
    torch.manual_seed(config.seed)

    # Cấu hình LoRA
    lora_config = LoraConfig(
        r=8,
        lora_alpha=16,
        lora_dropout=0.05,
        bias="none",
        task_type="CAUSAL_LM",
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj"]
    )

    # Tải mô hình chính với value head
    model = AutoModelForCausalLMWithValueHead.from_pretrained(
        model_name,
        torch_dtype=torch.bfloat16,  # Đồng bộ với reward model
        peft_config=lora_config,
        device_map="auto",
    )

    # Gán thủ công base_model_prefix để tránh lỗi
    model.base_model_prefix = "model"  # hoặc "transformer", tùy thuộc vào tên trong mô hình gốc
    # Gán generation_config để tránh lỗi AttributeError
    model.generation_config = GenerationConfig.from_pretrained(
        "./saved_models/lora-DeepSeek-R1-Distill-Qwen-adapter-merged"
    )
    

    # Tạo generation_config nếu không có
    if not hasattr(model, "generation_config"):
        model.generation_config = GenerationConfig.from_pretrained(model_name, trust_remote_code=True)

    print("Finetune model:", model_name, type(model))


    # Tạo optimizer (nếu dùng Adafactor)
    optimizer = None
    if script_args.adafactor:
        optimizer = Adafactor(
            filter(lambda p: p.requires_grad, model.parameters()),
            scale_parameter=False,
            relative_step=False,
            warmup_init=False,
            lr=config.learning_rate,
        )

    # Khởi tạo PPOTrainer
    print(dataset)
    ppo_trainer = PPOTrainer(
        config=config,  # Sử dụng args thay vì config
        model=model,
        tokenizer=tokenizer,  
        dataset=dataset,
        data_collator=collator,
        optimizer=optimizer,
    )
    

    # Xác định thiết bị
    device = ppo_trainer.accelerator.device
    if ppo_trainer.accelerator.num_processes == 1:
        device = 0 if torch.cuda.is_available() else "cpu"
    print("Device:", device)

    # Tạo sentiment pipeline
    sentiment_pipe = pipeline(
        "sentiment-analysis",
        model=reward_model_name,
        device_map="auto",
        # config=reward_model_config,
        tokenizer=tokenizer,
        # device=device,
    )

    # Cấu hình tham số sinh văn bản
    generation_kwargs = {
        "top_k": 0.0,
        "top_p": 1.0,
        "do_sample": True,
        "pad_token_id": tokenizer.pad_token_id,
        "eos_token_id": tokenizer.eos_token_id,
    }
    output_min_length = 32
    output_max_length = script_args.output_max_length
    output_length_sampler = LengthSampler(output_min_length, output_max_length)

    # Vòng lặp huấn luyện PPO
    for epoch, batch in tqdm(enumerate(ppo_trainer.dataloader)):
        question_tensors = batch["input_ids"]

        # Sinh phản hồi
        response_tensors = ppo_trainer.generate(
            question_tensors,
            return_prompt=False,
            length_sampler=output_length_sampler,
            **generation_kwargs,
        )
        batch["response"] = tokenizer.batch_decode(response_tensors, skip_special_tokens=True)

        # Tính điểm thưởng từ reward model
        texts = [q + r for q, r in zip(batch["query"], batch["response"])]
        pipe_outputs = sentiment_pipe(texts, **sent_kwargs)
        rewards = [torch.tensor(output[0]["score"] - script_args.reward_baseline) for output in pipe_outputs]

        # Thực hiện bước PPO
        stats = ppo_trainer.step(question_tensors, response_tensors, rewards)
        ppo_trainer.log_stats(stats, batch, rewards)

        # Lưu checkpoint định kỳ
        if script_args.save_freq and epoch and epoch % script_args.save_freq == 0:
            save_dir = os.path.join(script_args.output_dir, f"step_{epoch}")
            ppo_trainer.save_pretrained(save_dir)
            print(f"Saved checkpoint at: {save_dir}")

    # Lưu checkpoint cuối cùng
    final_save_dir = os.path.join(script_args.output_dir, "step_saved")
    ppo_trainer.save_pretrained(final_save_dir)
    print(f"Final checkpoint saved at: {final_save_dir}")
    
    # Xóa bộ nhớ sau mỗi bước
    torch.cuda.empty_cache()
    gc.collect()

# Chạy hàm
tuning_lm_with_rl(args)

# # Gộp adapter LoRA
from predict_module.merge_peft_adapter import merge_peft_adapter
merge_peft_adapter(
    model_name=os.path.join(args.output_dir, "step_saved"),
    output_name="./saved_models/sep_model"
)

## TEST

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from datasets import load_dataset
import pandas as pd
import re
import json

def split_completion(completion_text):
    if "Price Movement:" in completion_text:
        completion_text = completion_text.split("Price Movement:", -1)[-1]
        
    if "Explanation:" in completion_text:
        parts = completion_text.split("Explanation:", -1)

        match = re.search(r'(Positive|Negative|positive|negative)', parts[0])
        if match:
            target = match.group(1).capitalize()
        else:
            target = 'Mixed'

        explanation_raw = parts[1].strip()
        explain = "Explanation: " + re.split(r'\n|END OF EXAMPLES', explanation_raw)[0].strip()
    else:
        target = 'Mixed'
        explain = ""
        
    return target, explain

# Đường dẫn model và tokenizer
model_path = "./saved_models/sep_model" # output_name="./saved_models/sep_model"
tokenizer_name = "deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B"

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(tokenizer_name, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# Load model
model = AutoModelForCausalLM.from_pretrained(
    model_path,
    torch_dtype=torch.bfloat16,
    device_map="auto",
    trust_remote_code=True,
    local_files_only=True,
)

# Load dataset từ CSV KLTN/Data/summarized/OpenAILLM_top1_stock_data_test.csv
data_path = "../Data/summarized/OpenAILLM_top1_stock_data_test.csv"
test_ds = pd.read_csv(data_path)

# Chuẩn bị danh sách lưu kết quả
data_result = []

# Prompt template (bạn cần chắc chắn rằng PREDICT_INSTRUCTION và PREDICT_EXAMPLES đã được định nghĩa trước)

# Lặp qua từng mẫu dữ liệu
i = 1
for i, sample in test_ds.iterrows():
    print(f"ĐANG THỰC HIỆN MẪU i = {i}")
    i += 1

    ticker = sample["ticker"]
    summary = sample["summary"]
    label = sample["target"] #target

    prompt = PREDICT_INSTRUCTION.format(
        examples=PREDICT_EXAMPLES,
        ticker=ticker,
        summary=summary
    )

    # print("\n--- Prompt ---\n", prompt)

    # Tokenize prompt
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    # Generate response
    outputs = model.generate(
        **inputs,
        max_new_tokens=150,
        do_sample=True,
        top_p=0.95,
        top_k=0,
        temperature=0.6,
        pad_token_id=tokenizer.pad_token_id,
        eos_token_id=tokenizer.eos_token_id
    )

    # Decode và in ra response
    response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    print("\n--- Response ---\n", response)

    predict, explain = split_completion(response)

    data_result.append({
        "user_input": prompt,
        "prediction_of_LLM": predict,
        "explain": explain,
        "Label": label
    })

# Lưu kết quả ra file CSV
df = pd.DataFrame(data_result)
df.to_csv("SEP_RESULTS_top1_stock.csv", index=False, encoding="utf-8-sig")

# Dọn dẹp bộ nhớ GPU
torch.cuda.empty_cache()
from sklearn.metrics import accuracy_score, matthews_corrcoef

# Giả sử df là DataFrame của bạn
y_pred = df["prediction_of_LLM"]
y_true = df["Label"]

# Tính accuracy
acc = accuracy_score(y_true, y_pred)

# Tính MCC
mcc = matthews_corrcoef(y_true, y_pred)

print(f"Accuracy: {acc:.4f}")
print(f"MCC: {mcc:.4f}")